## Retrieval-Augmented Generation (RAG)

<a target="_blank" href="https://colab.research.google.com/github/microsoft/LLMLingua/blob/main/examples/RAG.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Retrieval-Augmented Generation (RAG) is a powerful and popular technique that applies specialized knowledge to large language models (LLMs). However, traditional RAG methods tend to have increasingly long prompts, sometimes exceeding **40k**, which can result in high financial and latency costs. Moreover, the decreased information density within the prompts can lead to performance degradation in LLMs, such as the "lost in the middle" issue.

<center><img width="800" src="https://github.com/microsoft/LLMLingua/blob/main/images/LongLLMLingua_Motivation.png?raw=1"></center>

To address this, we propose [**LongLLMLingua**](https://aclanthology.org/2024.acl-long.91/), which specifically tackles the low information density problem in long context scenarios via prompt compression, making it particularly suitable for RAG tasks. The main ideas involve a two-stage compression process, as shown by the  <font color='red'>**red line**</font>, which significantly improves the original curve:

- Coarse-grained compression through document-level perplexity;
- Fine-grained compression of the remaining text using token perplexity;

Instead of fighting against positional effects, we aim to utilize them to our advantage through document reordering, as illustrated by the  <font color='green'>**green line**</font>. In this approach, the most critical passages are placed at the beginning and the end of the context. Furthermore, the entire process becomes more **cost-effective and faster** since it only requires handling **1/4** of the original context.

### NaturalQuestions Multi-document QA

Next, we will demonstrate the use of LongLLMLingua on the NaturalQuestions dataset, which effectively alleviates the "lost in the middle" issue. This dataset closely resembles real-world RAG scenarios, as it first employs the Contriever retrieval system to recall 20 relevant documents (including 1 ground truth and 19 related documents), and then answers the respective questions based on the prompts composed of these 20 documents.

The original dataset can be found in https://github.com/nelson-liu/lost-in-the-middle/tree/main/qa_data.

In [1]:
# Install dependency.
## Lost in the middle
!git clone https://github.com/nelson-liu/lost-in-the-middle
!cd lost-in-the-middle && echo "xopen" > requirements.txt && pip install -e .
## LLMLingu
!pip install llmlingua
!pip install lost-in-the-middle

fatal: destination path 'lost-in-the-middle' already exists and is not an empty directory.


Obtaining file:///C:/Users/nanya/jupyter_bigdata/pm25um-big-data-main/lost-in-the-middle



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\python\python.exe -m pip install --upgrade pip
ERROR: file:///C:/Users/nanya/jupyter_bigdata/pm25um-big-data-main/lost-in-the-middle does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


^C


ERROR: Could not find a version that satisfies the requirement lost-in-the-middle (from versions: none)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\python\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for lost-in-the-middle


  Using cached llmlingua-0.2.2-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-1.11.0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.9.0-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached tiktoken-0.12.0-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   -------------------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\python\python.exe -m pip install --upgrade pip


In [7]:
# Using Google Gemini
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

In [ ]:
# Using Google Gemini (Recommended)
import google.generativeai as genai

# Replace with your actual Gemini API key
GEMINI_API_KEY = "your-gemini-api-key-here"
genai.configure(api_key=GEMINI_API_KEY)

### Setup Data

In [ ]:
# Setup Data for Medicine ETL
import pandas as pd
from lost_in_the_middle.prompting import Document

# Load medicine data from ETL output
medicine_df = pd.read_csv('/content/medicine_details_clean.csv')

# Create documents from medicine data
datasets = []
for idx, row in medicine_df.iterrows():
    # Create document text from medicine details
    doc_text = f"Medicine: {row['medicine_name']}\nComposition: {row['composition']}\nUses: {row['uses']}\nSide Effects: {row['side_effects']}\nManufacturer: {row['manufacturer']}"
    
    # Create a simple QA pair
    question = f"What are the uses of {row['medicine_name']}?"
    answer = row['uses']
    
    # Create documents list (using the medicine info as context)
    documents = [Document(title=row['medicine_name'], text=doc_text)]
    
    # Create prompt (simplified)
    instruction = "Answer the question based on the provided medicine information."
    demonstration = doc_text  # Use the document as demonstration
    question_part = question
    
    datasets.append({
        "id": idx,
        "instruction": instruction,
        "demonstration": demonstration,
        "question": question_part,
        "answer": [answer],  # List format like original
    })

print(f"Loaded {len(datasets)} medicine QA examples")

ModuleNotFoundError: No module named 'lost_in_the_middle'

In [ ]:
# select an example from NaturalQuestions
instruction, demonstration_str, question, answer = [
    datasets[23][key] for key in ["instruction", "demonstration", "question", "answer"]
]

In [ ]:
# Ground-truth Answer
answer

['14']

### The response of Original prompt (Error)

In [ ]:
# The response from original prompt using Gemini
prompt = "\n\n".join([instruction, demonstration_str, question])

model = genai.GenerativeModel('gemini-pro')
response = model.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        max_output_tokens=100,
        temperature=0,
        top_p=1,
    )
)
print("Response:", response.text)

{
    "id": "chatcmpl-8FFZIQCjv9Dv5Q9WQcDmNBT1OJIP8",
    "object": "chat.completion",
    "created": 1698645456,
    "model": "gpt-35-turbo",
    "choices": [
        {
            "index": 0,
            "finish_reason": "stop",
            "message": {
                "role": "assistant",
                "content": "As of the provided search results, OPEC has 15 member countries."
            }
        }
    ],
    "usage": {
        "prompt_tokens": 2897,
        "completion_tokens": 15,
        "total_tokens": 2912
    }
}


### The response of Compressed Prompt (Correct in 10x Compression)

In [ ]:
# Setup LLMLingua
from llmlingua import PromptCompressor

llm_lingua = PromptCompressor()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/hjiang/.local/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/hjiang/.local/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


In [ ]:
# 6x Compression with Gemini
compressed_prompt = llm_lingua.compress_prompt(
    demonstration_str.split("\n"),
    instruction=instruction,
    question=question,
    target_token=500,
    condition_compare=True,
    condition_in_question="after",
    rank_method="longllmlingua",
    use_sentence_level_filter=False,
    context_budget="+100",
    dynamic_context_compression_ratio=0.4,  # enable dynamic_context_compression_ratio
    reorder_context="sort",
)

model = genai.GenerativeModel('gemini-pro')
response = model.generate_content(
    compressed_prompt["compressed_prompt"],
    generation_config=genai.types.GenerationConfig(
        max_output_tokens=100,
        temperature=0,
        top_p=1,
    )
)

print(json.dumps(compressed_prompt, indent=4))
print("Response:", response.text)

{
    "compressed_prompt": "Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).\n\nDocument [10](Title: OPEC Organization of the Petroleum Exporting Countries (OPEC, /\u02c8o\u028ap\u025bk/ OH-pek, or OPEP in several other languages) is an intergovernmental organization of 14 nations as of February 2018, founded in 1960 in Baghdad by the first five members (Iran, Iraq, Kuwait, Saudi Arabia, and Venezuela), and headquartered since 1965 in Vienna, Austria. As of 2016, the 14 countries accounted for an estimated 44 percent of global oil production and 73 percent of the world's \"proven\" oil reserves, giving OPEC a major influence on global oil prices that were previously determined by American-dominated multinational oil companies.\n\nDocument1](Title: OPE OPE lost its newest members, who had in mid-1970s E withd in December 192, because it was unwilling to pay annual US$2 million membership fee felt that it neede

In [ ]:
# 10x Compression with Gemini
compressed_prompt = llm_lingua.compress_prompt(
    demonstration_str.split("\n"),
    instruction=instruction,
    question=question,
    target_token=100,
    condition_compare=True,
    condition_in_question="after",
    rank_method="longllmlingua",
    use_sentence_level_filter=False,
    context_budget="+100",
    dynamic_context_compression_ratio=0.4,  # enable dynamic_context_compression_ratio
    reorder_context="sort",
)

model = genai.GenerativeModel('gemini-pro')
response = model.generate_content(
    compressed_prompt["compressed_prompt"],
    generation_config=genai.types.GenerationConfig(
        max_output_tokens=100,
        temperature=0,
        top_p=1,
    )
)

print(json.dumps(compressed_prompt, indent=4))
print("Response:", response.text)

{
    "compressed_prompt": "Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).\n\n0Title: OPECization of Petroleum Exporting Count (OPEC, /\u02c8o\u028ap\u025bk OHpekP in other) is an intergovernmental14 nations as February 218 founded in 960 in Baghdad by fiveIran Iraq, Kuwait, Saudi Arab, and Venezuela), headquartered since 965 in, Austria. of the4ed an estimated4 percent of production and 3 percent of the world's \"proven\" oil res OPEC on global by Americandominatedin companies.\n\n5](Title: OPEC) OPEC lost its two newest members, who had joined in the mid-1970s. Ecuador withdrew in December 1992, because it was unwilling to pay the annual US$2 million membership fee and felt that it needed to produce more oil than it was allowed under the OPEC quota, although it rejoined in October 2007. Similar concerns prompted Gabon to suspend membership in January 1995; it rejoined in July 2016. Iraq has remained a mem

["# ETL Pipeline for Medicine Data from MongoDB", "", "This section implements an ETL (Extract, Transform, Load) pipeline based on the example from `UTS_BIGDATA.ipynb`, applied to extract medicine data from MongoDB and prepare it for analysis.", "", "### Pipeline Steps:", "- **Extract**: Connect to MongoDB and extract `medicine_details` and `medicine_usage` collections", "- **Transform**: Clean data (remove _id, standardize columns, handle duplicates)", "- **Load**: Create gold layer with master tables and quality reports", "", "Outputs will be saved to `etl_output/` folder."]

In [ ]:
["# Install required packages for ETL", "%pip install pymongo pandas numpy matplotlib seaborn --quiet"]

In [ ]:
["# Configuration for ETL Pipeline", "import pymongo", "import pandas as pd", "import os", "import numpy as np", "import matplotlib.pyplot as plt", "import seaborn as sns", "from datetime import datetime", "import logging", "import warnings", "", "# Configure warnings", "warnings.filterwarnings('ignore')", "", "# Configure logging", "logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')", "logger = logging.getLogger(__name__)", "", "# ETL Configuration", "ETL_CONFIG = {", "    'MONGODB_URI': 'mongodb://localhost:27017/',", "    'DATABASE_NAME': 'medicine_db',", "    'COLLECTIONS': {", "        'MEDICINE_DETAILS': 'medicine_details',", "        'MEDICINE_USAGE': 'medicine_usage'", "    },", "    'OUTPUT_DIR': 'etl_output/',", "    'FILES': {", "        'RAW_DETAILS': 'medicine_details_raw.csv',", "        'RAW_USAGE': 'medicine_usage_raw.csv',", "        'CLEAN_DETAILS': 'medicine_details_clean.csv',", "        'CLEAN_USAGE': 'medicine_usage_clean.csv',", "        'MASTER_DETAILS': 'medicine_master.csv',", "        'MASTER_USAGE': 'medicine_usage_master.csv',", "        'ETL_SUMMARY': 'etl_summary.csv',", "        'QUALITY_REPORT': 'data_quality_report.csv'", "    }", "}", "", "# Create output directory", "os.makedirs(ETL_CONFIG['OUTPUT_DIR'], exist_ok=True)", "print('ETL configuration loaded and output directory created')"]

In [ ]:
["# Utility Functions for ETL", "def connect_to_mongodb():", "    \"\"\"Establish connection to MongoDB\"\"\"", "    try:", "        client = pymongo.MongoClient(ETL_CONFIG['MONGODB_URI'])", "        db = client[ETL_CONFIG['DATABASE_NAME']]", "        # Test connection", "        db.list_collection_names()", "        logger.info(f'Connected to MongoDB: {ETL_CONFIG[\"DATABASE_NAME\"]}')", "        return db, client", "    except Exception as e:", "        logger.error(f'MongoDB connection failed: {str(e)}')", "        return None, None", "", "def save_dataframe(df, filename, description=''):", "    \"\"\"Save dataframe to CSV\"\"\"", "    try:", "        filepath = os.path.join(ETL_CONFIG['OUTPUT_DIR'], ETL_CONFIG['FILES'][filename])", "        df.to_csv(filepath, index=False)", "        logger.info(f'Saved {description}: {filepath}')", "        return True", "    except Exception as e:", "        logger.error(f'Error saving {description}: {str(e)}')", "        return False", "", "def clean_dataframe(df, name):", "    \"\"\"Basic data cleaning\"\"\"", "    df_clean = df.copy()", "    initial_rows = len(df_clean)", "    ", "    # Remove MongoDB ObjectId", "    if '_id' in df_clean.columns:", "        df_clean = df_clean.drop('_id', axis=1)", "    ", "    # Remove duplicates", "    df_clean = df_clean.drop_duplicates()", "    ", "    # Standardize column names", "    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')", "    ", "    final_rows = len(df_clean)", "    logger.info(f'{name} cleaned: {initial_rows} → {final_rows} rows')", "    return df_clean", "", "print('ETL utility functions loaded')"]

["## Extract Step: Connect to MongoDB and Extract Data"]

In [ ]:
["# Extract data from MongoDB", "db, client = connect_to_mongodb()", "", "if db is not None:", "    print(f'Connected to database: {db.name}')", "    collections = db.list_collection_names()", "    print(f'Available collections: {collections}')", "    ", "    # Extract data", "    collection_details = db[ETL_CONFIG['COLLECTIONS']['MEDICINE_DETAILS']]", "    collection_usage = db[ETL_CONFIG['COLLECTIONS']['MEDICINE_USAGE']]", "    ", "    df_medicine_details = pd.DataFrame(list(collection_details.find()))", "    df_medicine_usage = pd.DataFrame(list(collection_usage.find()))", "    ", "    print(f'Extracted {len(df_medicine_details):,} medicine details records')", "    print(f'Extracted {len(df_medicine_usage):,} medicine usage records')", "    ", "    # Save raw data", "    save_dataframe(df_medicine_details, 'RAW_DETAILS', 'Medicine Details (Raw)')", "    save_dataframe(df_medicine_usage, 'RAW_USAGE', 'Medicine Usage (Raw)')", "    ", "    print('Raw data saved to etl_output/')", "else:", "    print('Failed to connect to MongoDB. Please ensure MongoDB is running.')"]

["## Transform Step: Clean and Process Data"]

In [ ]:
["# Transform: Clean the data", "if 'df_medicine_details' in locals() and 'df_medicine_usage' in locals():", "    # Clean medicine details", "    df_medicine_details_clean = clean_dataframe(df_medicine_details, 'Medicine Details')", "    ", "    # Clean medicine usage", "    df_medicine_usage_clean = clean_dataframe(df_medicine_usage, 'Medicine Usage')", "    ", "    # Save cleaned data", "    save_dataframe(df_medicine_details_clean, 'CLEAN_DETAILS', 'Medicine Details (Cleaned)')", "    save_dataframe(df_medicine_usage_clean, 'CLEAN_USAGE', 'Medicine Usage (Cleaned)')", "    ", "    print('Data cleaning completed and saved to etl_output/')", "    ", "    # Preview cleaned data", "    print('\\nCleaned Medicine Details (first 3 rows):')", "    display(df_medicine_details_clean.head(3))", "    ", "    print('\\nCleaned Medicine Usage (first 3 rows):')", "    display(df_medicine_usage_clean.head(3))", "else:", "    print('No data to transform. Please run the Extract step first.')"]

["## Load Step: Create Gold Layer and Reports"]

In [ ]:
["# Load: Create gold layer and reports", "if 'df_medicine_details_clean' in locals() and 'df_medicine_usage_clean' in locals():", "    # Create master tables (gold layer)", "    save_dataframe(df_medicine_details_clean, 'MASTER_DETAILS', 'Medicine Master')", "    save_dataframe(df_medicine_usage_clean, 'MASTER_USAGE', 'Medicine Usage Master')", "    ", "    # Generate ETL summary", "    etl_summary = pd.DataFrame({", "        'metric': [", "            'Total Medicine Records',", "            'Total Usage Records',", "            'ETL Process Date',", "            'Status'", "        ],", "        'value': [", "            f\"{len(df_medicine_details_clean):,}\",", "            f\"{len(df_medicine_usage_clean):,}\",", "            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),", "            'SUCCESS'", "        ]", "    })", "    save_dataframe(etl_summary, 'ETL_SUMMARY', 'ETL Summary Report')", "    ", "    # Generate quality report", "    quality_report = pd.DataFrame({", "        'dataset': ['Medicine Details', 'Medicine Usage'],", "        'records': [len(df_medicine_details_clean), len(df_medicine_usage_clean)],", "        'data_quality_pct': [", "            ((1 - df_medicine_details_clean.isnull().sum().sum() / df_medicine_details_clean.size) * 100),", "            ((1 - df_medicine_usage_clean.isnull().sum().sum() / df_medicine_usage_clean.size) * 100)", "        ]", "    })", "    save_dataframe(quality_report, 'QUALITY_REPORT', 'Data Quality Report')", "    ", "    print('Gold layer and reports created successfully!')", "    print('\\nETL Summary:')", "    display(etl_summary)", "    print('\\nData Quality Report:')", "    display(quality_report)", "    ", "    # Close MongoDB connection", "    if 'client' in locals() and client is not None:", "        client.close()", "        print('\\nMongoDB connection closed.')", "else:", "    print('No cleaned data available. Please run Extract and Transform steps first.')"]

["## How to Run the ETL Pipeline", "", "### Prerequisites:", "- MongoDB running locally on `mongodb://localhost:27017/`", "- Database `medicine_db` with collections `medicine_details` and `medicine_usage`", "- Python packages installed (run the install cell first)", "", "### Steps to Execute:", "1. **Run Install Cell**: Install required packages", "2. **Run Configuration Cell**: Load config and create output directory", "3. **Run Extract Cell**: Connect to MongoDB and extract data", "4. **Run Transform Cell**: Clean and process the data", "5. **Run Load Cell**: Create gold layer and generate reports", "", "### Output Files in `etl_output/`:", "- `medicine_details_raw.csv` - Raw medicine details", "- `medicine_usage_raw.csv` - Raw medicine usage", "- `medicine_details_clean.csv` - Cleaned medicine details", "- `medicine_usage_clean.csv` - Cleaned medicine usage", "- `medicine_master.csv` - Gold layer medicine master", "- `medicine_usage_master.csv` - Gold layer usage master", "- `etl_summary.csv` - ETL process summary", "- `data_quality_report.csv` - Data quality assessment", "", "### Troubleshooting:", "- If MongoDB connection fails: Ensure MongoDB is running", "- If packages not found: Run `%pip install pymongo pandas numpy matplotlib seaborn`", "- If output directory issues: Check write permissions", "", "Run all cells in sequence to complete the ETL pipeline!"]